# Phase 4: Feature Engineering — Player Level

**Goal:** Given the 11 players in each team's XI (entered post-toss), compute player-level features and aggregate them to team level.

**Two-step approach:**
1. **Pre-compute:** Build a player stats lookup table from ball-by-ball history — stats as of a given date
2. **At prediction time:** User inputs 11 player names → system looks up their stats → aggregates to 8 team-level features

**Features we add (features 25–44):**
- Team batting: avg strike rate, avg batting average (top-order)
- Team bowling: avg economy, avg bowling SR (main bowlers)
- Experience: total IPL caps
- Star power: best batter, best bowler in XI
- Matchup features: batter vs bowler historical records

**Output:** `data/processed/player_career_stats.json` + `data/processed/full_features.csv`

## 1. Imports & Load

In [1]:
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

balls        = pd.read_csv('../data/processed/cleaned_balls.csv', parse_dates=['date'], low_memory=False)
team_features = pd.read_csv('../data/processed/team_features.csv', parse_dates=['date'])
player_df    = pd.read_csv('../data/raw/player_stats.csv')

print(f'Balls         : {len(balls):,}')
print(f'Team features : {len(team_features):,}')
print(f'Player stats  : {len(player_df):,}')

Balls         : 278,205
Team features : 1,169
Player stats  : 182


## 2. Compute Career Player Stats from Ball-by-Ball Data

We compute each player's stats from scratch using `cleaned_balls.csv`.
This gives us stats we can filter by date — critical for avoiding leakage.

**Why not just use `player_stats.csv`?**
Because it only has career totals with no dates — we can't ask
"what was Kohli's average as of April 2022?" from it.

In [2]:
# ── BATTING STATS ─────────────────────────────────────────────────────────
# valid_ball = 1 means it counts as a legal delivery faced by the batter
# (wides don't count as balls faced)

batting_stats = (
    balls.groupby('batter')
    .agg(
        batting_runs   = ('runs_batter', 'sum'),
        balls_faced    = ('valid_ball', 'sum'),
        innings_played = ('match_id', 'nunique'),
        dismissals     = ('is_wicket', 'sum'),   # times the batter got out
    )
    .reset_index()
)

# Strike Rate = (runs / balls) × 100
batting_stats['batting_sr'] = (
    batting_stats['batting_runs'] / batting_stats['balls_faced'].replace(0, np.nan) * 100
).round(2)

# Batting Average = runs / dismissals (how many runs per time out)
batting_stats['batting_avg'] = (
    batting_stats['batting_runs'] / batting_stats['dismissals'].replace(0, np.nan)
).round(2)

# Filter: only keep batters with at least 10 innings (unreliable below this)
batting_stats = batting_stats[batting_stats['innings_played'] >= 10].copy()

print(f'Batters with 10+ innings: {len(batting_stats)}')
batting_stats.sort_values('batting_runs', ascending=False).head(5)

Batters with 10+ innings: 335


,batter,batting_runs,balls_faced,innings_played,dismissals,batting_sr,batting_avg
658,V Kohli,8671,6505,259,231,133.30,37.54
500,RG Sharma,7048,5317,266,246,132.56,28.65
535,S Dhawan,6769,5304,221,194,127.62,34.89
151,DA Warner,6567,4680,184,164,140.32,40.04
569,SK Raina,5536,4025,200,168,137.54,32.95


In [3]:
# ── BOWLING STATS ─────────────────────────────────────────────────────────
bowling_stats = (
    balls.groupby('bowler')
    .agg(
        runs_conceded  = ('runs_bowler', 'sum'),
        balls_bowled   = ('valid_ball', 'sum'),
        wickets        = ('bowler_wicket', 'sum'),
        innings_bowled = ('match_id', 'nunique'),
    )
    .reset_index()
)

# Economy Rate = (runs / balls) × 6  (runs per over)
bowling_stats['bowling_econ'] = (
    bowling_stats['runs_conceded'] / bowling_stats['balls_bowled'].replace(0, np.nan) * 6
).round(2)

# Bowling Strike Rate = balls per wicket
bowling_stats['bowling_sr'] = (
    bowling_stats['balls_bowled'] / bowling_stats['wickets'].replace(0, np.nan)
).round(2)

# Filter: only keep bowlers with at least 10 innings
bowling_stats = bowling_stats[bowling_stats['innings_bowled'] >= 10].copy()

print(f'Bowlers with 10+ innings: {len(bowling_stats)}')
bowling_stats.sort_values('wickets', ascending=False).head(5)

Bowlers with 10+ innings: 291


,bowler,runs_conceded,balls_bowled,wickets,innings_bowled,bowling_econ,bowling_sr
543,YS Chahal,5032,3791,221,172,7.96,17.15
72,B Kumar,5412,4222,198,190,7.69,21.32
460,SP Narine,4933,4351,192,187,6.80,22.66
359,PP Chawla,5108,3850,192,191,7.96,20.05
369,R Ashwin,5652,4710,187,217,7.20,25.19


In [4]:
# ── CAPS (matches played) ─────────────────────────────────────────────────
# A player's IPL caps = number of unique matches they appeared in
# They appear in a match if they batted OR bowled
batters_per_match = balls[['match_id','batter']].drop_duplicates().rename(columns={'batter':'player'})
bowlers_per_match = balls[['match_id','bowler']].drop_duplicates().rename(columns={'bowler':'player'})
all_appearances   = pd.concat([batters_per_match, bowlers_per_match]).drop_duplicates()

caps = all_appearances.groupby('player')['match_id'].nunique().reset_index()
caps.columns = ['player', 'ipl_caps']

print(f'Players with caps data: {len(caps)}')
caps.sort_values('ipl_caps', ascending=False).head(5)

Players with caps data: 767


,player,ipl_caps
544,RG Sharma,266
718,V Kohli,260
535,RA Jadeja,246
421,MS Dhoni,241
314,KD Karthik,233


In [5]:
# ── MERGE ALL PLAYER STATS ────────────────────────────────────────────────
player_stats = (
    batting_stats[['batter','batting_runs','balls_faced','batting_sr','batting_avg','innings_played']]
    .rename(columns={'batter':'player'})
    .merge(
        bowling_stats[['bowler','wickets','bowling_econ','bowling_sr','innings_bowled']]
        .rename(columns={'bowler':'player'}),
        on='player', how='outer'
    )
    .merge(caps, on='player', how='outer')
)

# Fill NaN for players who only bat or only bowl
# Batting defaults: average SR ~120, average avg ~20
player_stats['batting_sr']  = player_stats['batting_sr'].fillna(120.0)
player_stats['batting_avg'] = player_stats['batting_avg'].fillna(20.0)
# Bowling defaults: average economy ~8.5 (IPL average), SR ~20
player_stats['bowling_econ'] = player_stats['bowling_econ'].fillna(8.5)
player_stats['bowling_sr']   = player_stats['bowling_sr'].fillna(20.0)
player_stats['ipl_caps']     = player_stats['ipl_caps'].fillna(0).astype(int)

print(f'Total players in lookup table: {len(player_stats)}')
player_stats.head()

Total players in lookup table: 767


,player,batting_runs,balls_faced,batting_sr,batting_avg,innings_played,wickets,bowling_econ,bowling_sr,innings_bowled,ipl_caps
0,A Ashish Reddy,280.0,193.0,145.08,18.67,23.0,18.0,9.07,14.56,20.0,28
1,A Badoni,963.0,693.0,138.96,26.75,46.0,NaN,8.50,20.00,NaN,48
2,A Chandila,NaN,NaN,120.00,20.00,NaN,11.0,6.21,21.27,12.0,12
3,A Chopra,NaN,NaN,120.00,20.00,NaN,NaN,8.50,20.00,NaN,6
4,A Choudhary,NaN,NaN,120.00,20.00,NaN,NaN,8.50,20.00,NaN,5


In [6]:
# Verify: Kohli and Bumrah stats should match what we know
print('Kohli:')
print(player_stats[player_stats['player'] == 'V Kohli'][['player','batting_runs','batting_sr','batting_avg']].to_string(index=False))

print('\nBumrah:')
print(player_stats[player_stats['player'] == 'JJ Bumrah'][['player','wickets','bowling_econ','bowling_sr']].to_string(index=False))

Kohli:
 player  batting_runs  batting_sr  batting_avg
V Kohli        8671.0       133.3        37.54

Bumrah:
   player  wickets  bowling_econ  bowling_sr
JJ Bumrah    186.0          7.25       18.06


In [7]:
# Save as JSON for fast lookup at prediction time
# Key = player name, Value = their stats dict
stats_dict = player_stats.set_index('player').to_dict(orient='index')

with open('../data/processed/player_career_stats.json', 'w') as f:
    json.dump(stats_dict, f, indent=2, default=str)

print(f'Saved player_career_stats.json — {len(stats_dict)} players')
print('Sample entry (V Kohli):', json.dumps(stats_dict.get('V Kohli', {}), indent=2))

Saved player_career_stats.json — 767 players
Sample entry (V Kohli): {
  "batting_runs": 8671.0,
  "balls_faced": 6505.0,
  "batting_sr": 133.3,
  "batting_avg": 37.54,
  "innings_played": 259.0,
  "wickets": 4.0,
  "bowling_econ": 8.8,
  "bowling_sr": 62.75,
  "innings_bowled": 26.0,
  "ipl_caps": 260
}


## 3. Batter vs Bowler Matchup Features

For every (batter, bowler) pair in history, compute:
- How many balls faced
- Runs scored
- Strike rate in that matchup
- Dismissal rate

This is what TV broadcast **matchup cards** are based on.

In [8]:
# Compute batter vs bowler stats across all historical deliveries
# Only count valid balls (wides don't count)
matchup_df = (
    balls[balls['valid_ball'] == 1]
    .groupby(['batter', 'bowler'])
    .agg(
        balls_faced   = ('valid_ball', 'sum'),
        runs_scored   = ('runs_batter', 'sum'),
        dismissals    = ('bowler_wicket', 'sum'),
    )
    .reset_index()
)

# Only keep matchups with at least 6 balls — below this, stats aren't reliable
matchup_df = matchup_df[matchup_df['balls_faced'] >= 6].copy()

matchup_df['matchup_sr']            = (matchup_df['runs_scored'] / matchup_df['balls_faced'] * 100).round(2)
matchup_df['matchup_dismissal_rate'] = (matchup_df['dismissals'] / matchup_df['balls_faced']).round(4)

print(f'Matchup pairs (6+ balls): {len(matchup_df):,}')

# Famous example: check Kohli vs a spinner
kohli_matchups = matchup_df[matchup_df['batter'] == 'V Kohli'].sort_values('balls_faced', ascending=False)
print('\nKohli top matchups by balls faced:')
print(kohli_matchups.head(5).to_string(index=False))

Matchup pairs (6+ balls): 14,660

Kohli top matchups by balls faced:
 batter    bowler  balls_faced  runs_scored  dismissals  matchup_sr  matchup_dismissal_rate
V Kohli RA Jadeja          160          179           3      111.87                  0.0188
V Kohli  R Ashwin          149          181           1      121.48                  0.0067
V Kohli SP Narine          129          136           4      105.43                  0.0310
V Kohli KH Pandya          107          117           1      109.35                  0.0093
V Kohli PP Chawla          105          140           3      133.33                  0.0286


In [9]:
# Save matchup table
matchup_df.to_csv('../data/processed/player_matchups.csv', index=False)
print(f'Saved player_matchups.csv — {len(matchup_df):,} rows')

Saved player_matchups.csv — 14,660 rows


## 4. Extract Playing XIs from Historical Matches

For training: derive who played in each historical match from the ball-by-ball data.
Players who batted OR bowled in a match = part of the XI.

In [10]:
# Get all batters per (match_id, team)
batters_in_match = (
    balls[['match_id', 'batting_team', 'batter']]
    .drop_duplicates()
    .rename(columns={'batting_team': 'team', 'batter': 'player'})
)

# Get all bowlers per (match_id, team) — bowler belongs to the bowling_team
bowlers_in_match = (
    balls[['match_id', 'bowling_team', 'bowler']]
    .drop_duplicates()
    .rename(columns={'bowling_team': 'team', 'bowler': 'player'})
)

# Combine: a player is in the XI if they batted or bowled
xi_df = pd.concat([batters_in_match, bowlers_in_match]).drop_duplicates()

print(f'Total player-match appearances: {len(xi_df):,}')
print(f'Avg players per team per match: {len(xi_df) / (len(balls["match_id"].nunique()) * 2):.1f}')

Total player-match appearances: 25,018


TypeError: object of type 'int' has no len()

## 5. Aggregate Player Stats to Team Level per Match

For each match, take the XI players and compute team-level aggregates.
This is the set of features the model actually uses.

In [ ]:
def aggregate_xi_stats(match_id, team, player_stats_df, xi_df, matchup_df, opp_team=None):
    """
    Given a match_id and team, get their XI players and compute team-level features.
    player_stats_df : precomputed career stats lookup
    xi_df           : who played in each match
    matchup_df      : batter vs bowler historical matchups
    opp_team        : the opposing team (for matchup features)
    """
    # Get this team's players in this match
    players = xi_df[(xi_df['match_id'] == match_id) & (xi_df['team'] == team)]['player'].tolist()

    if len(players) == 0:
        return {}

    # Look up stats for each player
    team_player_stats = player_stats_df[player_stats_df['player'].isin(players)]

    # ── BATTING features ─────────────────────────────────────────────────
    avg_batting_sr  = team_player_stats['batting_sr'].mean()
    avg_batting_avg = team_player_stats['batting_avg'].mean()
    best_batting_avg = team_player_stats['batting_avg'].max()

    # ── BOWLING features ─────────────────────────────────────────────────
    avg_bowling_econ = team_player_stats['bowling_econ'].mean()
    avg_bowling_sr   = team_player_stats['bowling_sr'].mean()
    best_bowling_econ = team_player_stats['bowling_econ'].min()  # lower = better

    # ── EXPERIENCE ───────────────────────────────────────────────────────
    total_caps = team_player_stats['ipl_caps'].sum()

    # ── MATCHUP features ─────────────────────────────────────────────────
    favorable_matchups = 0
    avg_matchup_sr = avg_batting_sr  # default to team avg SR if no matchup data

    if opp_team is not None:
        # Get opposition bowlers
        opp_players = xi_df[(xi_df['match_id'] == match_id) & (xi_df['team'] == opp_team)]['player'].tolist()

        # Find matchups between this team's batters and opp bowlers
        matchups = matchup_df[
            (matchup_df['batter'].isin(players)) &
            (matchup_df['bowler'].isin(opp_players))
        ]

        if len(matchups) > 0:
            # Favorable = batter has SR > 140 against that bowler
            favorable_matchups = (matchups['matchup_sr'] > 140).sum()
            avg_matchup_sr = matchups['matchup_sr'].mean()

    return {
        'avg_batting_sr'   : round(avg_batting_sr, 2),
        'avg_batting_avg'  : round(avg_batting_avg, 2),
        'best_batting_avg' : round(best_batting_avg, 2),
        'avg_bowling_econ' : round(avg_bowling_econ, 2),
        'avg_bowling_sr'   : round(avg_bowling_sr, 2),
        'best_bowling_econ': round(best_bowling_econ, 2),
        'total_caps'       : int(total_caps),
        'favorable_matchups': int(favorable_matchups),
        'avg_matchup_sr'   : round(avg_matchup_sr, 2),
    }

In [ ]:
# Test on one match before running the full loop
sample_match = team_features.iloc[100]
sample_id    = sample_match['match_id']
sample_ta    = sample_match['team_a']
sample_tb    = sample_match['team_b']

ta_stats = aggregate_xi_stats(sample_id, sample_ta, player_stats, xi_df, matchup_df, opp_team=sample_tb)
tb_stats = aggregate_xi_stats(sample_id, sample_tb, player_stats, xi_df, matchup_df, opp_team=sample_ta)

print(f'Match: {sample_ta} vs {sample_tb}')
print(f'{sample_ta} stats: {ta_stats}')
print(f'{sample_tb} stats: {tb_stats}')

In [ ]:
from tqdm.notebook import tqdm

player_feature_rows = []

for _, match_row in tqdm(team_features.iterrows(), total=len(team_features), desc='Player features'):
    mid = match_row['match_id']
    ta  = match_row['team_a']
    tb  = match_row['team_b']

    ta_stats = aggregate_xi_stats(mid, ta, player_stats, xi_df, matchup_df, opp_team=tb)
    tb_stats = aggregate_xi_stats(mid, tb, player_stats, xi_df, matchup_df, opp_team=ta)

    row = {'match_id': mid}
    for k, v in ta_stats.items():
        row[f'ta_{k}'] = v
    for k, v in tb_stats.items():
        row[f'tb_{k}'] = v

    player_feature_rows.append(row)

player_features = pd.DataFrame(player_feature_rows)
print(f'\nDone! Shape: {player_features.shape}')
player_features.head()

## 6. Merge Team + Player Features → Full Feature Set

In [ ]:
# Merge team features (Phase 3) with player features (Phase 4)
full_features = team_features.merge(player_features, on='match_id', how='left')

# Fill any remaining nulls with column medians
# (happens when a team's XI has no players in our stats table)
feature_cols = [c for c in full_features.columns
                if c not in ['match_id','date','season_year','team_a','team_b','venue','team_a_won']]
for col in feature_cols:
    full_features[col] = full_features[col].fillna(full_features[col].median())

print(f'Full features shape : {full_features.shape}')
print(f'Total feature cols  : {len(feature_cols)}')
print(f'Nulls remaining     : {full_features.isnull().sum().sum()}')
print(f'\nAll feature columns:')
print(feature_cols)

## 7. Verify Key Stats

In [ ]:
print('=== SANITY CHECKS ===')

# Kohli SR should be ~133
kohli = player_stats[player_stats['player'] == 'V Kohli'].iloc[0]
print(f'Kohli SR: {kohli["batting_sr"]} (expect ~133)')
print(f'Kohli avg: {kohli["batting_avg"]} (expect ~36-40)')

# Bumrah economy should be ~7.4
bumrah = player_stats[player_stats['player'] == 'JJ Bumrah'].iloc[0]
print(f'Bumrah econ: {bumrah["bowling_econ"]} (expect ~7.2-7.5)')
print(f'Bumrah wickets: {bumrah["wickets"]} (expect 180+)')

# team avg SR should be between 100-150 for all matches
print(f'\nta_avg_batting_sr range: {full_features["ta_avg_batting_sr"].min():.1f} – {full_features["ta_avg_batting_sr"].max():.1f}')
print(f'ta_avg_bowling_econ range: {full_features["ta_avg_bowling_econ"].min():.1f} – {full_features["ta_avg_bowling_econ"].max():.1f}')

## 8. Save Full Feature Set

In [ ]:
full_features.to_csv('../data/processed/full_features.csv', index=False)
print(f'Saved full_features.csv — {full_features.shape[0]} rows × {full_features.shape[1]} columns')
print(f'Ready for model training in Phase 6.')

## Your Tasks (Before Phase 5)

```
□ Look up your favourite player's computed stats:
  player_stats[player_stats['player'] == 'YOUR PLAYER NAME']
  Do the numbers match what you know?

□ Look up a famous matchup — e.g. Rohit Sharma vs a spinner:
  matchup_df[(matchup_df['batter']=='RG Sharma') & (matchup_df['bowler']=='Harbhajan Singh')]

□ Understand WHY we use avg batting SR not just total runs
  (what does a higher SR mean for a T20 team?)

□ Understand WHY we take best_bowling_econ as min() not mean()
  (what does having one elite bowler in the XI mean?)

□ Add to interview notebook:
  Q: 'How do you handle players not in your training data?'
  A: [your answer — hint: look at the fillna defaults]

  Q: 'What is a player matchup feature? Give a real example.'
  A: [your answer]

  Q: 'Why compute player stats from ball-by-ball instead of using the pre-made CSV?'
  A: [your answer]
```